# Dataset Provenance: Mouse × Indicator × Experiment × Sessions × Neurons

This notebook compiles the dataset provenance tables for the manuscript Methods section.

It builds:
1. **Summary Table by Indicator & Experiment** (without mouse names)
2. **Compact Experiment Summary** (without mouse names)
3. **Cross-Experiment Mouse Matrix** (Table 3)
4. **Dedicated Experiment Tables** for:
   - Closed loop
   - Treadmill (motorized wheel)
   - Open loop
   - SF/TF (drifting gratings)
   - Multi-day tracking
   - Multi-depth RF
5. Prints the LaTeX of Table 3 and of the compact experiment summary, to paste into Overleaf.

In [ ]:
%reload_ext autoreload
%autoreload 2

from IPython.display import display, Markdown

from v1_depth_map.paths import get_figures_roots
from v1_depth_map.figure_utils.build_methods_dataset_tables import (
    EXPERIMENT_LABELS,
    get_flexilims_sessions,
    load_datasets,
    mouse_table,
    sex_string,
    build_anonymous_summary_table,
    build_anonymous_compact_table,
    build_cross_experiment_matrix,
    generate_latex_matrix,
    generate_latex_compact,
)


In [ ]:
flexilims_sessions = get_flexilims_sessions()
READ_ROOT, _ = get_figures_roots(flexilims_sessions["Original"])

In [ ]:
# Neuron counts per session: cached Fig 1 / Fig 2 / Fig S10 population files and curated
# cells of the revision sessions. Sex from flexilims.
df_sessions = load_datasets(READ_ROOT, flexilims_sessions)
mice = mouse_table(df_sessions)
print(f"{len(mice)} mice in total: {sex_string(mice['sex'], long=True)}")
print(f"Loaded {len(df_sessions)} session records across {df_sessions['experiment_key'].nunique()} experiments.")
print(f"Unique sessions: {df_sessions['session'].nunique()}, total neuron entries: {df_sessions['neurons'].sum():,}")


## 1. Summary Tables (Without Mouse Names)

In [ ]:
anon_summary_df = build_anonymous_summary_table(df_sessions)
display(Markdown("### Summary by Experiment & Indicator"))
display(anon_summary_df[["experiment", "indicator", "genotype", "mice", "sex", "sessions", "neurons"]])

anon_compact_df = build_anonymous_compact_table(df_sessions)
display(Markdown("### Compact Summary by Experiment"))
display(anon_compact_df)


## 2. Table 3: Cross-Experiment Mouse Matrix

In [ ]:
matrix_df = build_cross_experiment_matrix(df_sessions)
display(matrix_df.drop(columns=["Cohort"]))


## 3. Dedicated Tables for Each Experiment

In [ ]:
for exp_k, exp_name in EXPERIMENT_LABELS.items():
    sub = df_sessions[df_sessions["experiment_key"] == exp_k]
    exp_summary = (
        sub.groupby(["mouse", "indicator", "sex"], as_index=False)
        .agg(sessions=("session", "nunique"), neurons=("neurons", "sum"))
        .sort_values("mouse")
    )
    display(Markdown(f"### {exp_name} ({exp_summary['sessions'].sum()} sessions, {exp_summary['neurons'].sum():,} neurons)"))
    display(exp_summary)


## 4. LaTeX of Table 3

In [ ]:
# Standard \hline rules, no booktabs package required (use_booktabs=True for booktabs rules)
print(generate_latex_matrix(matrix_df, use_booktabs=False))

## 5. LaTeX of the Compact Summary by Experiment

In [ ]:
# Same rule options as Table 3 (use_booktabs=True for booktabs rules)
print(generate_latex_compact(anon_compact_df, use_booktabs=False))